In [ ]:
import weaviate
from weaviate.classes.init import AdditionalConfig, Timeout
client = weaviate.connect_to_local(additional_config=AdditionalConfig(
        timeout=Timeout(init=30, query=4800, insert=120)  # Values in seconds
    ))
collection = client.collections.get("Articles")

## Bielik RAG z Weaviate 

Wykonajmy najpierw `bm25` z użyciem modelu szukając danych w pełnym tekście u używając zapytania `Wajda rozdanie oscarów` a następnie znaleziony tekst w bazie wektorowej dodajmy do prompta `Na podstawie tekstu ponizej wylistuj rywali Pana Tadeusza.\n\n{pelen_tekst}`

💡 Weaviate automatycznie wyszuka dany artykuł w bazie i doda pierwszy wynik do promptu wysyłanego do `Bielika`

💡 To zapytanie może potrwać kilka minut gdyż uruchamiamy Bielika lokalnie z użyciem Ollama w dockerze

In [ ]:
result = collection.generate.bm25(
    query="Wajda rozdanie oscarów",
    query_properties=["pelen_tekst"],
    limit=1,
    single_prompt="Na podstawie tekstu ponizej wylistuj rywali Pana Tadeusza.\n\n{pelen_tekst}",
)
for obj in result.objects:
    print(f"-{obj.generated}\n")

Wykonajmy jeszcze jedno zapytanie ale używając obu indeksów wektorowych (obu modeli AI)

💡 Weaviate automatycznie zwektoryzuje query: `Czy AI zniszczy ludzkość?` z użyciem 2-óch wektoryzerów:
- `silver_retriever` model: [ipipan/silver-retriever-base-v1.1](https://huggingface.co/ipipan/silver-retriever-base-v1.1).
- `bge_m3` model: [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

połączy wyniki i doda pierwszy wynik do promptu wysyłanego do `Bielika`

💡 To zapytanie może potrwać kilka minut gdyż uruchamiamy Bielika lokalnie z użyciem Ollama w dockerze

In [ ]:
from weaviate.classes.query import TargetVectors

result = collection.generate.near_text(
    query="Wpływ telefonów komórkowych na zdrowie",
    target_vector=TargetVectors.sum(["silver_retriever", "bge_m3"]),
    limit=1,
    single_prompt="Wyłącznie na podstawie podanego kontekstu odpowiedz zwięźle na pytanie: 'Czy telefony komórkowe niszczą zdrowie?'\n{pelen_tekst}"
)
for obj in result.objects:
    print(f"-{obj.generated}\n")

In [ ]:
client.close()